# Analyse de sentiment des avis d'hôtels TripAdvisor
Objectif : prédire la note (1 à 5 étoiles) d'un avis à partir de son texte, et comprendre ce qui rend les clients mécontents.

## 0. Chargement et vérification des données

In [1]:
from datasets import load_dataset

ds = load_dataset("nhull/tripadvisor-split-dataset-v2")

### Les splits

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['review', 'label'],
        num_rows: 30400
    })
    validation: Dataset({
        features: ['review', 'label'],
        num_rows: 1600
    })
    test: Dataset({
        features: ['review', 'label'],
        num_rows: 8000
    })
})

Il y a 40 000 avis en tout (30 400 train, 1 600 validation, 8 000 test), alors que la description du dataset en annonce 80 000.

### Conversion en DataFrames

In [3]:
df_train = ds["train"].to_pandas()
df_val = ds["validation"].to_pandas()
df_test = ds["test"].to_pandas()

df_train.head()
#df_train.shape

,review,label
0,"Good room, dodgy area\nNothing at all wrong wi...",3.0
1,Great Hotel\nStayed three days over the 4th of...,4.0
2,"Online rating was 3-star, I would give it 1-st...",1.0
3,Great place to stay on PB!\nThis is the perfec...,5.0
4,"Good Value, Great location\nThe Ramada Gaslamp...",4.0


In [4]:
print(df_train["review"].iloc[0])

Good room, dodgy area
Nothing at all wrong with the hotel. Good staff with large rooms and comfortable beds. The location is not ideal and you have to walk through a lot of human detrius to get into town. We thought the tarriff was high considering the area. We later stayed in San francisco in a cheaper hotel in a better location.


### Équilibre des classes

In [5]:
#les classes sont-elles vraiment équilibrées dans chaque split? ;
print(df_train['label'].value_counts().sort_index())
print(df_val['label'].value_counts().sort_index())
print(df_test['label'].value_counts().sort_index())

label
1.0    6080
2.0    6080
3.0    6080
4.0    6080
5.0    6080
Name: count, dtype: int64
label
1.0    320
2.0    320
3.0    320
4.0    320
5.0    320
Name: count, dtype: int64
label
1.0    1600
2.0    1600
3.0    1600
4.0    1600
5.0    1600
Name: count, dtype: int64


Chaque note représente exactement 20 % des avis dans chaque split (6 080 avis par note dans le train, 320 en validation, 1 600 dans le test). Les classes sont donc équilibrées et il n’y a pas besoin de rééchantillonnage (resampling) pour l’entraînement.

Mais il faut noter que cet équilibre est artificiel : sur TripAdvisor, la majorité des avis sont à 4 ou 5 étoiles et les avis à 1 étoile sont moins courants (dans la première version de ce projet, 74 % des avis étaient positifs). → Les performances mesurées sur ce test équilibré seront probablement un peu optimistes : en conditions réelles, comme les avis négatifs sont plus rares, une plus grande part des avis signalés comme négatifs seront de fausses alertes.

### Type des colonnes

In [6]:
df_val.info()

<class 'pandas.DataFrame'>
RangeIndex: 1600 entries, 0 to 1599
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   review  1600 non-null   str    
 1   label   1600 non-null   float64
dtypes: float64(1), str(1)
memory usage: 1.5 MB


In [ ]:
#on vérifie qu'il n'y a pas de nan
valeur_unique_train = df_train["label"].unique()
valeur_unique_val = df_val["label"].unique()
valeur_unique_test = df_test["label"].unique()

print(valeur_unique_train, valeur_unique_val, valeur_unique_test)

[3. 4. 1. 5. 2.] [5. 1. 2. 3. 4.] [4. 1. 2. 3. 5.]


In [8]:
df_train["label"] = df_train["label"].astype(int)
df_val["label"] = df_val["label"].astype(int)
df_test["label"] = df_test["label"].astype(int)
print(df_train["label"].dtype, df_val["label"].dtype, df_test["label"].dtype)

int64 int64 int64


Label était stocké en float (1.0 à 5.0), alors qu’une note ne prend que des valeurs entières. Après vérification avec .unique(), je l’ai converti en int : c’est plus lisible.

## 1. Exploration du texte
### A. Lecture d'exemples

In [15]:
for note in range(1, 6):
    print(f"========== {note} étoile(s) ==========")
    exemples = df_train[df_train["label"] == note].sample(2, random_state=42)
    for avis in exemples["review"]:
        print(avis)
        print("---")

========== 1 étoile(s) ==========
Pleeeze renovate me! YUCK!
Oh, man .... Everything worn to threads, carpet threadbare, wallpaper torn, bathroom tile cracked. Mold in bathroom. Furniture dinged up, cracked. Nice new tv though.
Paid extra for balcony facing river and we looked out onto a rooftop full of broken beer bottles.
I could go on and on but instead just let me say .... Don't stay here even if it's a low rate. We arrived too late to change hotels but didn't stay our second night as planned.
---
Surprisingly Bad
As a platinum with Starwood, I've seen a lot of their properties all over the world. On the whole they're pretty good, which is why I usually Starwood when I have the opportunity. And they usually reward my loyalty with pretty good service and cool benefits like free breakfast and room upgrades.
This particular one, though, is one of the really terrible ones. I arrived late, around 11:30 p.m., and was told my room wasn't available. This is surprising because usually my ro

### Observations (sur 10 avis lus, 2 par note)

- **Structure** : chaque avis commence par un **titre** (la première ligne, séparée du texte par un retour à la ligne `\n`). Le titre résume l'avis général du client et peut aider à prédire la note, mais il peut aussi être trompeur : un titre poli et positif peut précéder un texte plein de reproches.
  → Idée d'expérience : comparer un modèle qui lit seulement le titre avec un modèle qui lit l'avis complet.

- **Forme du texte** : la ponctuation est correcte et les retours à la ligne sont utilisés dans certains avis seulement. Les majuscules suivent la structure normale des phrases (début de phrase, noms propres) ; elles ne servent pas à exprimer une émotion (pas de « GREAT » ou « NEVER AGAIN » dans les exemples lus).
  → Pour TF-IDF, on pourra passer le texte en minuscules sans perdre d'information. Pour VADER, on gardera le texte brut, car il tient compte de la ponctuation et des majuscules.

- **Chiffres** : ils indiquent surtout des horaires ou des durées (heure d'arrivée, nombre de nuits, temps d'attente). Exception notable : certains clients écrivent une note dans le texte (« Online rating was 3-star, I would give it 1-star »).
  → Ces mentions de note sont un indice très fort pour le modèle ; à surveiller quand on analysera les mots qu'il utilise.

- **Langue** : tous les avis lus sont en anglais (cohérent avec l'origine du dataset, filtré sur l'anglais, voir ci-dessous).

- **Vocabulaire** : les avis positifs contiennent des adjectifs mélioratifs (*clean, fast, spacious, best*) et les avis négatifs des adjectifs péjoratifs (*bad, dirty, tedious, disorganised*).
  → À vérifier par la suite : le modèle retrouve-t-il ces mots parmi les plus importants ? Attention aux négations (« not clean » contient « clean »).

- **Avis à 3 étoiles** : dans les exemples lus, ils ressemblent davantage à des avis à 4 étoiles qu'à des avis à 2 étoiles.
  → Hypothèse à vérifier avec la matrice de confusion : le modèle confondra-t-il plus souvent 3 ★ avec 4 ★ qu'avec 2 ★ ?

- **Cohérence note / texte** : les 10 avis lus correspondaient à leur note. Mais 10 avis sur 30 400 ne suffisent pas pour conclure : la page du dataset précise que la note ne reflète pas toujours le contenu du texte.
  → Les avis incohérents seront étudiés lors de l'analyse d'erreurs.

### Origine du dataset

Ce dataset ([nhull/tripadvisor-split-dataset-v2](https://huggingface.co/datasets/nhull/tripadvisor-split-dataset-v2)) a été construit pour un cours universitaire d'NLP : 40 000 avis en anglais, sélectionnés pour avoir autant d'avis de chaque note, puis découpés en train / validation / test. Il est issu d'un dataset plus large ([jniimi/tripadvisor-review-rating](https://huggingface.co/datasets/jniimi/tripadvisor-review-rating), 201 000 avis) qui contient aussi l'identifiant de l'hôtel, les dates et des notes par aspect (propreté, rapport qualité-prix, emplacement, chambres, sommeil).

→ Ce dataset est adapté pour **entraîner et comparer des modèles de classification**, mais pas pour mesurer la fréquence réelle des problèmes rencontrés par les hôtels. Pour l'analyse métier, le dataset d'origine (distribution réaliste, notes par aspect) sera plus pertinent, en excluant les avis utilisés pour l'entraînement afin d'éviter toute fuite de données.

In [ ]:
res = df_train["review"].str.contains(r"\b[A-Z]{3,}\b")
res.mean()

np.float64(0.32457236842105264)